# Bradford Bulls — Team-Aware Frame Extraction v3.0

### Pipeline
```
Phase 0A: Detect static overlays (scoreboard, watermark)
Phase 0B: Semi-supervised team calibration (YOU label a few samples)
Pass 1:  Fast scan → find quality segments
Pass 2:  Team-aware extraction + quota selection
Save:    Frames + enriched metadata CSV
```

**New in v3**: Instead of unreliable auto-clustering, you now label 3-5 jersey
samples from a numbered grid. The system learns from YOUR examples.

## 0. Install Dependencies
Run this cell, then **Restart Runtime**.

In [ ]:
!pip install -q ultralytics>=8.3.0 scikit-learn imagehash scikit-image pillow opencv-python-headless roboflow
print("\nDone. Please RESTART RUNTIME now (Runtime > Restart runtime).")

## 1. YOUR CONFIG

In [ ]:
MEMBER_NAME = input("Your name (e.g. edward): ").strip().lower()
VIDEO_FILENAME = input("Video filename (e.g. M06_black_1080p.mp4): ").strip()
test_input = input("Test mode? (y/n, default=y): ").strip().lower()
TEST_MODE = test_input != "n"

mode_label = "TEST" if TEST_MODE else "PRODUCTION"
print(f"\n--- Config ---")
print(f"  Member:   {MEMBER_NAME}")
print(f"  Video:    {VIDEO_FILENAME}")
print(f"  Mode:     {mode_label}")


## 2. Setup Environment

In [ ]:
import torch, cv2, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected.")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/Bradford_Bulls_AI")
VIDEOS_DIR = DRIVE_ROOT / "videos"
FRAMES_DIR = DRIVE_ROOT / "frames"
METADATA_DIR = DRIVE_ROOT / "metadata"
REPORTS_DIR = DRIVE_ROOT / "reports"

for d in [VIDEOS_DIR, FRAMES_DIR, METADATA_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MATCH_ID = Path(VIDEO_FILENAME).stem
VIDEO_PATH = VIDEOS_DIR / VIDEO_FILENAME
FRAMES_OUTPUT_DIR = FRAMES_DIR / MATCH_ID
FRAMES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
print(f"\nVideo: {VIDEO_PATH.name} ({VIDEO_PATH.stat().st_size/1e6:.0f} MB)")


## 2.5 Standalone Backend Modules

The cells below define helper / overlay / calibration / pipeline / selection
modules inline. They share the notebook's global scope, so each module can
freely reference functions defined in earlier cells.


### Module: helpers

Sharpness, pitch detection, timestamps, YOLO person detection.


In [ ]:
# helpers — primitives used by every downstream module
import cv2
import numpy as np


def fmt_timestamp(sec):
    """Convert seconds to HH:MM:SS string for filenames."""
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}h{m:02d}m{s:02d}s" if h > 0 else f"{m:02d}m{s:02d}s"


def compute_sharpness(region, mask=None):
    """Multi-metric sharpness score in [0, 1]. Higher = sharper.

    When `mask` is supplied (1 = clean pixel, 0 = overlay), overlay pixels
    are excluded so scoreboards don't inflate the score.
    """
    gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY) if len(region.shape) == 3 else region

    if mask is not None:
        valid = mask.astype(bool)
        if valid.sum() < 100:
            return 0.0
    else:
        valid = np.ones(gray.shape, dtype=bool)

    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_var = float(lap[valid].var()) if valid.any() else 0.0

    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    ten_vals = (gx ** 2 + gy ** 2)[valid]
    ten = float(ten_vals.mean()) if len(ten_vals) > 0 else 0.0

    score = min(lap_var / 300, 1.0) * 0.55 + min(ten / 5000, 1.0) * 0.45
    return round(score, 4)


def compute_player_sharpness(frame, detections, overlay_mask=None):
    """Max sharpness across detected persons, computed per-bbox with overlay masking."""
    if not detections:
        return compute_sharpness(frame, overlay_mask)

    h, w = frame.shape[:2]
    scores = []

    for det in detections:
        x1, y1, x2, y2 = [int(v) for v in det["bbox"]]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        crop_mask = None
        if overlay_mask is not None:
            crop_mask = overlay_mask[y1:y2, x1:x2]
            # Skip crops where >60% is overlay (graphics, not player).
            if crop_mask.mean() < 0.4:
                continue

        scores.append(compute_sharpness(crop, crop_mask))

    return round(max(scores), 4) if scores else 0.0


def compute_pitch_green_ratio(frame_bgr, roi_y_start=0.40):
    """Fraction of grass-green pixels in the bottom ROI ([0, 1])."""
    h, w = frame_bgr.shape[:2]
    y0 = int(max(0, min(h - 1, roi_y_start * h)))
    roi = frame_bgr[y0:h, 0:w]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    lower = np.array([35, 40, 40])
    upper = np.array([95, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    return float(mask.mean() / 255.0)


def smart_pitch_filter(pitch_green_ratio, detections, frame_area,
                       min_green=0.06, closeup_bypass_ratio=0.10):
    """Bypass the pitch-green check for close-up shots (they're the most valuable)."""
    if detections:
        max_area = max(d["area"] for d in detections)
        if max_area / frame_area > closeup_bypass_ratio:
            return True
    return pitch_green_ratio >= min_green


def filter_foreground_players(detections, frame_h, frame_w):
    """Drop background crowd detections; keep significant-sized standing persons."""
    if not detections or len(detections) < 2:
        return detections

    areas = [d["area"] for d in detections]
    median_area = np.median(areas)

    foreground = []
    for d in detections:
        x1, y1, x2, y2 = d["bbox"]
        bh = y2 - y1

        is_significant = d["area"] > median_area * 0.25
        is_tall_enough = bh > frame_h * 0.05

        if is_significant and is_tall_enough:
            d["is_foreground"] = True
            foreground.append(d)
        else:
            d["is_foreground"] = False

    return foreground if foreground else [detections[0]]


def detect_persons(yolo_model, frame, device, confidence=0.5):
    """YOLO person detection on a single frame; returns list of bbox dicts."""
    results = yolo_model.predict(
        frame, classes=[0], conf=confidence,
        device=device, verbose=False
    )
    detections = []
    if results and results[0].boxes is not None:
        xyxy = results[0].boxes.xyxy.cpu().numpy()
        confs = results[0].boxes.conf.cpu().numpy()
        for i in range(len(xyxy)):
            x1, y1, x2, y2 = xyxy[i]
            detections.append({
                "bbox": [float(x1), float(y1), float(x2), float(y2)],
                "conf": float(confs[i]),
                "area": float((x2 - x1) * (y2 - y1)),
            })
    return detections


def get_shot_type(max_person_area_ratio):
    """Classify shot type from the largest person size."""
    if max_person_area_ratio > 0.12:
        return "closeup"
    elif max_person_area_ratio > 0.05:
        return "medium"
    else:
        return "wide"


def is_graphics_frame(frame_bgr,
                     pitch_green_safe=0.20,
                     min_saturated_fraction=0.20,
                     dominant_non_grass_share=0.30):
    """Heuristic: True if the frame is dominated by TV graphics (replay, intro card).

    Logic:
      1. If pitch_green_ratio > pitch_green_safe → definitely a real game frame.
      2. Otherwise, look at high-saturation pixels (graphics are saturated).
         If fewer than min_saturated_fraction → natural scene (crowd, sky), keep.
      3. Build hue histogram of saturated pixels. Mask out the grass band.
         If one non-grass hue bin holds > dominant_non_grass_share of the
         saturated pixels → graphics background, reject.
    """
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)

    green = cv2.inRange(hsv, (35, 40, 40), (85, 255, 255))
    pitch_ratio = float(green.mean() / 255.0)
    if pitch_ratio > pitch_green_safe:
        return False

    h_chan = hsv[..., 0]
    s_chan = hsv[..., 1]
    sat_mask = s_chan > 80

    total_px = frame_bgr.shape[0] * frame_bgr.shape[1]
    if sat_mask.sum() < total_px * min_saturated_fraction:
        return False

    hist, _ = np.histogram(h_chan[sat_mask], bins=18, range=(0, 180))
    # Grass-green sits roughly H 40-90 → bins 4..8 in 18 bins of width 10.
    hist_non_grass = hist.copy()
    hist_non_grass[4:9] = 0
    total_sat = int(hist.sum())
    if total_sat == 0:
        return False
    top_share = float(hist_non_grass.max()) / total_sat
    return top_share > dominant_non_grass_share


# pHash helpers (for frame-similarity dedup).
from PIL import Image as _PIL_Image
import imagehash as _imagehash


def compute_phash(frame_bgr, region_norm=None, size=256):
    """Perceptual hash of a frame (or normalized sub-region of it).

    Args:
        frame_bgr: full BGR frame.
        region_norm: optional (x1, y1, x2, y2) in [0, 1] — hash only this crop.
        size: square resize for the hashed image.

    Returns:
        imagehash.ImageHash. Use `(a - b)` for Hamming distance.
    """
    if region_norm is not None:
        h, w = frame_bgr.shape[:2]
        x1, y1, x2, y2 = region_norm
        x1, x2 = sorted((max(0, int(x1 * w)), min(w, int(x2 * w))))
        y1, y2 = sorted((max(0, int(y1 * h)), min(h, int(y2 * h))))
        if x2 - x1 < 8 or y2 - y1 < 8:
            target = frame_bgr
        else:
            target = frame_bgr[y1:y2, x1:x2]
    else:
        target = frame_bgr
    small = cv2.resize(target, (size, size))
    rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    return _imagehash.phash(_PIL_Image.fromarray(rgb))



### Module: overlay

Detect static broadcast overlays (scoreboard, watermark) via temporal variance.


In [ ]:
# overlay — temporal-variance overlay detection
import cv2
import numpy as np


def detect_static_overlays(video_path, n_samples=30):
    """Build a binary overlay mask from per-pixel temporal variance.

    Pixels that barely change across N sampled frames are treated as
    overlay (scoreboard, watermark). Only edge regions are considered
    because broadcast overlays always live near the borders.

    Returns:
        overlay_mask: (H, W) uint8, 1 = clean pixel, 0 = overlay pixel
        overlay_ratio: fraction of the frame flagged as overlay
    """
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

    start = int(total * 0.05)
    end = int(total * 0.95)
    indices = np.linspace(start, end, n_samples, dtype=int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            small = cv2.resize(frame, (w // 2, h // 2))
            frames.append(small.astype(np.float32))
    cap.release()

    if len(frames) < 5:
        print("WARNING: Too few frames for overlay detection. Skipping.")
        return np.ones((h, w), dtype=np.uint8), 0.0

    stacked = np.stack(frames)
    variance = stacked.var(axis=0).mean(axis=2)

    # Bottom-3% of variance values flagged as static (adaptive, floor=8.0).
    thresh = max(np.percentile(variance, 3), 8.0)
    static_mask_small = (variance < thresh).astype(np.uint8)

    # Restrict to the outer band — overlays never sit in the field center.
    sh, sw = static_mask_small.shape
    edge_mask = np.zeros_like(static_mask_small)
    edge_mask[:int(sh * 0.18), :] = 1
    edge_mask[int(sh * 0.80):, :] = 1
    edge_mask[:, :int(sw * 0.22)] = 1
    edge_mask[:, int(sw * 0.78):] = 1
    static_mask_small = static_mask_small * edge_mask

    kernel = np.ones((11, 11), np.uint8)
    static_mask_small = cv2.morphologyEx(static_mask_small, cv2.MORPH_CLOSE, kernel)
    static_mask_small = cv2.morphologyEx(static_mask_small, cv2.MORPH_OPEN, kernel)

    static_mask_full = cv2.resize(static_mask_small, (w, h), interpolation=cv2.INTER_NEAREST)

    overlay_mask = 1 - static_mask_full
    overlay_ratio = 1.0 - overlay_mask.mean()

    return overlay_mask, overlay_ratio


def visualize_overlay(frame, overlay_mask):
    """Render detected overlay regions in red over the original frame."""
    import matplotlib.pyplot as plt

    vis = frame.copy()
    vis[overlay_mask == 0] = [0, 0, 255]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original Frame")
    axes[0].axis("off")

    blend = cv2.addWeighted(frame, 0.6, vis, 0.4, 0)
    axes[1].imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Overlay regions (red) — {(1-overlay_mask.mean())*100:.1f}% masked")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


def build_manual_overlay_mask(frame_shape, regions, normalized=True):
    """Build an overlay mask from a list of bounding boxes the user provided.

    Args:
        frame_shape: shape tuple of the target frame ((H, W) or (H, W, C)).
        regions: iterable of (x1, y1, x2, y2). Each rectangle is MASKED OUT.
        normalized: if True, coords are fractions in [0, 1] of frame size.
                    If False, coords are absolute pixels.

    Returns:
        overlay_mask: (H, W) uint8 where 1 = clean pixel, 0 = overlay pixel.
        overlay_ratio: fraction of pixels marked as overlay.
    """
    h, w = frame_shape[:2]
    mask = np.ones((h, w), dtype=np.uint8)
    for box in regions:
        x1, y1, x2, y2 = box
        if normalized:
            x1, y1, x2, y2 = x1 * w, y1 * h, x2 * w, y2 * h
        xa = max(0, min(w, int(round(min(x1, x2)))))
        xb = max(0, min(w, int(round(max(x1, x2)))))
        ya = max(0, min(h, int(round(min(y1, y2)))))
        yb = max(0, min(h, int(round(max(y1, y2)))))
        if xb > xa and yb > ya:
            mask[ya:yb, xa:xb] = 0
    overlay_ratio = 1.0 - float(mask.mean())
    return mask, overlay_ratio


def visualize_overlay_with_grid(frame, overlay_mask, grid_step=0.1):
    """Visualize overlay mask with a normalized [0, 1] coordinate grid.

    Use this to read off (x1, y1, x2, y2) values for OVERLAY_REGIONS.
    """
    import matplotlib.pyplot as plt

    h, w = frame.shape[:2]
    vis = frame.copy()
    vis[overlay_mask == 0] = [0, 0, 255]
    blend = cv2.addWeighted(frame, 0.55, vis, 0.45, 0)

    fig, ax = plt.subplots(figsize=(15, 8))
    ax.imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB))

    n_steps = int(round(1.0 / grid_step))
    for i in range(n_steps + 1):
        x_px = i * grid_step * w
        y_px = i * grid_step * h
        ax.axvline(x_px, color="yellow", alpha=0.35, linewidth=0.6)
        ax.axhline(y_px, color="yellow", alpha=0.35, linewidth=0.6)
        ax.text(x_px, -8, f"{i*grid_step:.1f}",
                color="orange", fontsize=8, ha="center", va="bottom")
        ax.text(-8, y_px, f"{i*grid_step:.1f}",
                color="orange", fontsize=8, ha="right", va="center")

    ax.set_title(
        f"Manual overlay mask — {(1 - overlay_mask.mean()) * 100:.1f}% masked.\n"
        f"Read normalized (x, y) from the yellow grid (0,0 top-left → 1,1 bottom-right).",
        fontsize=11,
    )
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def combine_overlay_masks(*masks):
    """Logical AND of multiple overlay masks (any mask saying 'overlay' wins)."""
    if not masks:
        raise ValueError("combine_overlay_masks needs at least one mask")
    out = masks[0].copy()
    for m in masks[1:]:
        out = np.minimum(out, m)
    return out, 1.0 - float(out.mean())


def manual_box_input(prompt="Enter boxes", single_box=False):
    """Text-input fallback when the interactive picker is unavailable.
    Each line: x1,y1,x2,y2  (normalized 0-1). Empty line ends the list."""
    print(prompt)
    print("Enter boxes as comma-separated normalized coords, one per line.")
    print("Example: 0.78,0.00,1.00,0.18")
    print("Empty line to finish.")
    boxes = []
    while True:
        line = input(f"Box {len(boxes)+1} (empty to stop): ").strip()
        if not line:
            break
        try:
            parts = [float(x) for x in line.split(",")]
            if len(parts) != 4:
                print("  Need exactly 4 values.")
                continue
            boxes.append(tuple(parts))
            if single_box:
                break
        except ValueError:
            print("  Invalid number format, try again.")
    if single_box:
        return boxes[0] if boxes else None
    return boxes

def interactive_box_picker(frame, prompt="Drag to draw rectangles. Click 'Done' when finished.",
                           single_box=False, max_display_width=1100,
                           max_iframe_height=720):
    """Interactive rectangle picker for Colab.

    UI layout: sticky top bar with Done/Undo/Clear buttons (always visible),
    scrollable canvas below. Iframe height is capped so the cell output area
    creates its own scroll if needed.
    """
    import base64
    try:
        from google.colab.output import eval_js
    except ImportError:
        print("google.colab unavailable - using text-input fallback.")
        return manual_box_input(prompt, single_box=single_box)

    ok, buf = cv2.imencode(".png", frame)
    if not ok:
        raise RuntimeError("Failed to encode frame for picker")
    b64 = base64.b64encode(buf.tobytes()).decode("ascii")
    h, w = frame.shape[:2]

    disp_w = min(w, max_display_width)
    disp_h = int(h * disp_w / w)
    iframe_h = min(disp_h + 110, max_iframe_height)
    single = "true" if single_box else "false"
    prompt_safe = prompt.replace("\\", "\\\\").replace("'", "\\'").replace("\n", " ")

    js_template = r"""
    (async function() {
      try {
        if (window.google && window.google.colab && window.google.colab.output) {
          window.google.colab.output.setIframeHeight(__IFRAME_H__, true);
        }
      } catch (e) { console.warn('setIframeHeight failed:', e); }

      return await new Promise((resolve) => {
        // Inject CSS once.
        if (!document.getElementById('ovl-picker-style')) {
          const style = document.createElement('style');
          style.id = 'ovl-picker-style';
          style.textContent = `
            .ovl-wrap { font-family: sans-serif; border: 1px solid #ddd;
              border-radius: 6px; background: #fafafa; margin: 8px 0; }
            .ovl-bar { position: sticky; top: 0; z-index: 100;
              background: #fafafa; padding: 10px 12px;
              border-bottom: 1px solid #ccc;
              display: flex; gap: 8px; align-items: center; flex-wrap: wrap; }
            .ovl-bar button { padding: 6px 14px; cursor: pointer;
              border: 1px solid #999; background: white; border-radius: 4px;
              font-size: 13px; }
            .ovl-bar button#ovl_done { background: #2c7; color: white;
              border: 0; font-weight: 600; padding: 8px 18px; }
            .ovl-bar button#ovl_done:disabled { background: #888; cursor: default; }
            .ovl-prompt { padding: 8px 12px; font-weight: 600; color: #333;
              border-bottom: 1px solid #eee; }
            .ovl-canvas-wrap { padding: 10px; overflow: auto; }
            .ovl-canvas-wrap canvas { display: block; cursor: crosshair;
              border: 1px solid #888; max-width: 100%; height: auto;
              user-select: none; }
            .ovl-hint { color: #666; font-size: 12px; margin-left: 8px; }
          `;
          document.head.appendChild(style);
        }

        const wrap = document.createElement('div');
        wrap.className = 'ovl-wrap';
        wrap.innerHTML = (
          '<div class="ovl-bar">' +
          '  <button id="ovl_done">Done</button>' +
          '  <button id="ovl_undo">Undo last</button>' +
          '  <button id="ovl_clear">Clear all</button>' +
          '  <span class="ovl-hint" id="ovl_count">0 rectangles</span>' +
          '</div>' +
          '<div class="ovl-prompt">__PROMPT__</div>' +
          '<div class="ovl-canvas-wrap">' +
          '  <canvas id="ovl_cv" width="__W__" height="__H__"></canvas>' +
          '</div>'
        );
        document.body.appendChild(wrap);

        const cv = wrap.querySelector('#ovl_cv');
        const ctx = cv.getContext('2d');
        const img = new Image();
        let imgLoaded = false;
        img.onload = () => { imgLoaded = true; redraw(); };
        img.onerror = (e) => { console.error('Image load failed', e); };
        img.src = 'data:image/png;base64,__B64__';

        let rects = [];
        let drawing = false;
        let startPt = null;
        const single = __SINGLE__;

        function redraw(extra) {
          if (!imgLoaded) return;
          ctx.drawImage(img, 0, 0);
          ctx.strokeStyle = '#ff2222';
          ctx.lineWidth = 4;
          for (const r of rects) {
            ctx.strokeRect(r[0], r[1], r[2]-r[0], r[3]-r[1]);
          }
          if (extra) {
            ctx.strokeStyle = '#ffaa00';
            ctx.lineWidth = 3;
            ctx.strokeRect(extra[0], extra[1], extra[2]-extra[0], extra[3]-extra[1]);
          }
          wrap.querySelector('#ovl_count').textContent = rects.length + ' rectangles';
        }

        function ptOf(e) {
          const r = cv.getBoundingClientRect();
          return [(e.clientX - r.left) * cv.width / r.width,
                  (e.clientY - r.top) * cv.height / r.height];
        }

        cv.addEventListener('mousedown', (e) => { e.preventDefault(); startPt = ptOf(e); drawing = true; });
        cv.addEventListener('mousemove', (e) => {
          if (!drawing) return;
          const c = ptOf(e);
          redraw([Math.min(startPt[0], c[0]), Math.min(startPt[1], c[1]),
                  Math.max(startPt[0], c[0]), Math.max(startPt[1], c[1])]);
        });
        cv.addEventListener('mouseup', (e) => {
          if (!drawing) return;
          const end = ptOf(e); drawing = false;
          const x1 = Math.min(startPt[0], end[0]), y1 = Math.min(startPt[1], end[1]);
          const x2 = Math.max(startPt[0], end[0]), y2 = Math.max(startPt[1], end[1]);
          if ((x2-x1) > 4 && (y2-y1) > 4) {
            if (single) rects = [[x1, y1, x2, y2]];
            else rects.push([x1, y1, x2, y2]);
          }
          redraw();
        });
        cv.addEventListener('mouseleave', () => { drawing = false; redraw(); });

        wrap.querySelector('#ovl_undo').onclick = () => { rects.pop(); redraw(); };
        wrap.querySelector('#ovl_clear').onclick = () => { rects = []; redraw(); };
        wrap.querySelector('#ovl_done').onclick = () => {
          const W = cv.width, H = cv.height;
          const norm = rects.map(r => [r[0]/W, r[1]/H, r[2]/W, r[3]/H]);
          const btn = wrap.querySelector('#ovl_done');
          btn.textContent = 'Submitted';
          btn.disabled = true;
          resolve(norm);
        };
      });
    })();
    """
    js = (js_template
          .replace("__IFRAME_H__", str(iframe_h))
          .replace("__PROMPT__", prompt_safe)
          .replace("__W__", str(w))
          .replace("__H__", str(h))
          .replace("__B64__", b64)
          .replace("__SINGLE__", single))

    print("[picker] Active - the Done/Undo/Clear bar is at the TOP of the canvas.")
    print("[picker] Scroll inside the picker area if needed; 30-min timeout.")

    try:
        result = eval_js(js, timeout_sec=1800)
    except Exception as e:
        print(f"[picker] Interactive picker failed: {e}")
        print("[picker] Falling back to text input...")
        return manual_box_input(prompt, single_box=single_box)

    if result is None:
        result = []
    boxes = [tuple(r) for r in result]
    print(f"[picker] Got {len(boxes)} box(es).")

    if single_box:
        return boxes[0] if boxes else None
    return boxes



### Module: calibration

Semi-supervised team color calibration. User labels a handful of jersey
crops; pipeline learns a k-NN reference set per team.


In [ ]:
# calibration - semi-supervised team color classification
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# ═══════════════════════════════════════════════════════════════════════════
# Feature extraction
# ═══════════════════════════════════════════════════════════════════════════

def _build_green_mask(hsv_img):
    """Binary mask: 1 = non-green pixel, 0 = grass/green."""
    green = cv2.inRange(hsv_img, (35, 40, 40), (85, 255, 255))
    return (green == 0).astype(np.float32)


def _build_gaussian_weights(h, w, sigma=0.4):
    """2D Gaussian centered on crop — emphasizes jersey center, suppresses edges."""
    y = np.linspace(-1, 1, h)
    x = np.linspace(-1, 1, w)
    yy, xx = np.meshgrid(y, x, indexing='ij')
    return np.exp(-0.5 * (xx**2 + yy**2) / sigma**2).astype(np.float32)


def extract_torso_features(torso_crop, overlay_crop_mask=None):
    """
    Compute a 300-dim weighted HSV histogram for the torso crop.

    Weights combine three masks:
      1. Gaussian center emphasis (edges less important)
      2. Green exclusion (grass pixels zeroed out)
      3. Overlay exclusion (scoreboard/text pixels zeroed out)

    This ensures text overlays like "LDN", "BRA 4th" etc. don't
    contaminate the jersey color signature.
    """
    if torso_crop is None or torso_crop.size < 200:
        return None

    hsv = cv2.cvtColor(torso_crop, cv2.COLOR_BGR2HSV)
    h, w = torso_crop.shape[:2]

    # Combined weight: center emphasis × green exclusion × overlay exclusion
    weights = _build_gaussian_weights(h, w, 0.4) * _build_green_mask(hsv)

    if overlay_crop_mask is not None:
        # overlay_crop_mask: 1=clean pixel, 0=overlay pixel
        # Resize to match torso crop if needed
        if overlay_crop_mask.shape[:2] != (h, w):
            overlay_crop_mask = cv2.resize(
                overlay_crop_mask.astype(np.float32), (w, h),
                interpolation=cv2.INTER_NEAREST
            )
        weights *= overlay_crop_mask.astype(np.float32)

    if weights.sum() < 50:
        return None

    pixels = hsv.reshape(-1, 3).astype(np.float64)
    w_flat = weights.flatten()

    hist, _ = np.histogramdd(
        pixels, bins=[12, 5, 5],
        range=[(0, 180), (0, 256), (0, 256)],
        weights=w_flat
    )
    hist = hist.flatten()
    norm = np.linalg.norm(hist)
    if norm > 0:
        hist /= norm
    return hist.astype(np.float64)


# ═══════════════════════════════════════════════════════════════════════════
# Torso crop extraction
# ═══════════════════════════════════════════════════════════════════════════

def extract_torso_crop(frame, bbox, overlay_mask=None, strict=False):
    """
    Extract the jersey region from a YOLO person bounding box.

    Filters applied:
      - Too small (bbox < 40×60 px)
      - Close-up / staff (bbox > 20% of frame = likely coach/commentator)
      - Bad aspect ratio (width > height = not a standing player)
      - Overlay contaminated (scoreboard/graphics region)
      - Mostly skin (shirtless or wrong crop)
      - Mostly grass (misaligned crop)
      - Too blurry (only in strict/calibration mode)

    Args:
        strict: True during calibration (rejects blurry); False at runtime
    Returns:
        (torso_crop, status_string)
        When overlay_mask is provided and the crop passes, the torso's
        overlay mask slice is stored as torso_crop._overlay_mask_crop
        (a numpy array) for use in feature extraction.
    """
    fh, fw = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    bw, bh = x2 - x1, y2 - y1

    if bw < 40 or bh < 60:
        return None, "too_small", None

    # Close-up filter: large bbox = likely TV close-up of staff/coach
    if (bw * bh) / (fh * fw) > 0.20:
        return None, "close_up", None

    # Standing player check: height should be > width
    if bh / max(bw, 1) < 1.0:
        return None, "bad_aspect", None

    # Jersey ROI: 10-40% of person height
    ty1 = max(0, y1 + int(bh * 0.10))
    ty2 = min(fh, y1 + int(bh * 0.40))
    tx1 = max(0, x1 + int(bw * 0.10))
    tx2 = min(fw, x2 - int(bw * 0.10))

    if ty2 - ty1 < 15 or tx2 - tx1 < 15:
        return None, "crop_too_small", None

    torso = frame[ty1:ty2, tx1:tx2]

    # Overlay check — reject crops with >25% overlay contamination
    # (stricter than before: old threshold was 50%)
    torso_overlay_crop = None
    if overlay_mask is not None:
        torso_overlay_crop = overlay_mask[ty1:ty2, tx1:tx2]
        clean_ratio = torso_overlay_crop.mean()
        if clean_ratio < 0.75:
            return None, "overlay", None

    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)

    # Skin check
    skin = cv2.inRange(hsv, (0, 30, 60), (25, 180, 255))
    if skin.mean() / 255 > 0.6:
        return None, "mostly_skin", None

    # Grass check
    grass = cv2.inRange(hsv, (35, 40, 40), (85, 255, 255))
    if grass.mean() / 255 > 0.5:
        return None, "mostly_grass", None

    # Sharpness (calibration only)
    if strict:
        gray = cv2.cvtColor(torso, cv2.COLOR_BGR2GRAY)
        if cv2.Laplacian(gray, cv2.CV_64F).var() < 100:
            return None, "blurry", None

    return torso, "ok", torso_overlay_crop


# ═══════════════════════════════════════════════════════════════════════════
# Step 1: Collect diverse samples
# ═══════════════════════════════════════════════════════════════════════════

def collect_samples(video_path, yolo_model, device, overlay_mask=None,
                    n_sample_frames=80, n_display=24):
    """
    Collect torso crops from the video and select a diverse subset for labeling.
    
    Returns a dict containing:
      - 'all_crops': list of all valid torso crops (64×64 BGR)
      - 'all_features': list of feature vectors
      - 'display_indices': indices of the diverse subset to show the user
      - 'display_crops': the crops to display (ordered by diversity)
    """
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_area = fw * fh

    start = int(total * 0.05)
    end = int(total * 0.95)
    indices = np.linspace(start, end, n_sample_frames, dtype=int)

    all_crops, all_features = [], []
    reject_counts = {}

    print(f"Sampling {n_sample_frames} frames for calibration...")
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        results = yolo_model.predict(frame, classes=[0], conf=0.5,
                                     device=device, verbose=False)
        if not results or results[0].boxes is None:
            continue

        for box in results[0].boxes:
            bbox = box.xyxy[0].cpu().numpy()
            area = float((bbox[2] - bbox[0]) * (bbox[3] - bbox[1]))
            if area / frame_area < 0.02:
                continue

            torso, status, torso_ov_mask = extract_torso_crop(
                frame, bbox, overlay_mask, strict=True)
            if status != "ok":
                reject_counts[status] = reject_counts.get(status, 0) + 1
                continue

            feat = extract_torso_features(torso, torso_ov_mask)
            if feat is None:
                continue

            all_crops.append(cv2.resize(torso, (64, 64)))
            all_features.append(feat)

    cap.release()
    n_total = len(all_crops)
    print(f"Collected {n_total} valid torso crops.")
    if reject_counts:
        print(f"  Filtered out: {reject_counts}")

    if n_total < 10:
        print("ERROR: Too few crops. Try lowering YOLO confidence or min_person_area_ratio.")
        return None

    # ── Select diverse subset for display ──
    # Use mini K-Means to pick maximally diverse samples
    X = np.array(all_features)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    n_show = min(n_display, n_total)
    if n_total <= n_display:
        display_indices = list(range(n_total))
    else:
        # Pick diverse samples: run K-Means with K=n_display, 
        # then pick the sample closest to each centroid
        km = KMeans(n_clusters=n_show, random_state=42, n_init=5)
        km.fit(X_scaled)
        display_indices = []
        for i in range(n_show):
            cluster_members = np.where(km.labels_ == i)[0]
            center = km.cluster_centers_[i]
            dists = np.linalg.norm(X_scaled[cluster_members] - center, axis=1)
            display_indices.append(int(cluster_members[dists.argmin()]))

    return {
        "all_crops": all_crops,
        "all_features": all_features,
        "scaler": scaler,
        "X_scaled": X_scaled,
        "display_indices": display_indices,
        "n_total": n_total,
    }


# ═══════════════════════════════════════════════════════════════════════════
# Step 2: Show numbered grid
# ═══════════════════════════════════════════════════════════════════════════

def show_samples(sample_data):
    """
    Display a numbered grid of diverse torso crop samples.
    The user will pick which numbers belong to their team.
    """
    if sample_data is None:
        print("ERROR: No sample data. Run collect_samples() first.")
        return

    indices = sample_data["display_indices"]
    crops = sample_data["all_crops"]
    n = len(indices)

    # 4 columns layout
    cols = min(6, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.5 * rows))
    if rows == 1:
        axes = axes[np.newaxis, :]
    if cols == 1:
        axes = axes[:, np.newaxis]

    for i in range(rows * cols):
        ax = axes[i // cols, i % cols]
        if i < n:
            idx = indices[i]
            crop = crops[idx]
            ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            # Big clear number
            ax.set_title(f"#{i}", fontsize=16, fontweight='bold',
                        color='white',
                        bbox=dict(boxstyle='round,pad=0.3', 
                                  facecolor='black', alpha=0.8))
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle(
        f"PICK YOUR TEAM — Write down the numbers of YOUR team's jerseys\n"
        f"(Total {sample_data['n_total']} crops sampled from video)",
        fontsize=15, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 60)
    print("  📋 In the NEXT cell, set MY_TEAM to a list of numbers")
    print("     that show YOUR team's jersey. Example:")
    print("     MY_TEAM = [0, 3, 5, 8, 11]")
    print("=" * 60)


# ═══════════════════════════════════════════════════════════════════════════
# Step 3: Build calibration from user labels
# ═══════════════════════════════════════════════════════════════════════════

def build_calibration(sample_data, my_team_indices,
                      classify_margin=0.58, ref_confidence_min=0.66):
    """Build a calibration model from user-labeled samples.

    Strategy (negative-signal k-NN with margin):
      1. Picked display crops          → target_refs_seed
      2. NOT-picked display crops      → opponent_refs_seed
      3. For every crop, compute d_target / d_opponent vs both seed sets.
         confidence = 1 - d_min / (d_t + d_o + eps)
      4. Label as 0/1 only when confidence >= classify_margin. Otherwise -1
         (ambiguous) — these crops do NOT get assigned to either team.
      5. Final refs use confidently classified crops (>= ref_confidence_min)
         plus all explicitly user-labeled crops.

    Args:
        classify_margin: minimum confidence to assign a label (0.5-1.0).
            0.58 means d_winning < ~0.72 * d_other. Higher = stricter.
        ref_confidence_min: minimum confidence to admit a crop to the final
            reference set. Should be > classify_margin.

    Returns:
        calibration dict consumed by classify_person().
    """
    if sample_data is None:
        print("ERROR: No sample data.")
        return None

    display_indices = sample_data["display_indices"]
    X_scaled = sample_data["X_scaled"]
    scaler = sample_data["scaler"]
    crops = sample_data["all_crops"]
    n_total = sample_data["n_total"]

    target_grid_set = set(my_team_indices)
    target_crop_indices = set()
    skipped_grid = []
    for grid_num in my_team_indices:
        if 0 <= grid_num < len(display_indices):
            target_crop_indices.add(display_indices[grid_num])
        else:
            skipped_grid.append(grid_num)
    if skipped_grid:
        print(f"  WARN: out-of-range grid numbers ignored: {skipped_grid}")

    if len(target_crop_indices) < 2:
        print("ERROR: Need at least 2 labeled samples. Try again.")
        return None

    opponent_seed_crop_indices = set()
    for grid_num, crop_idx in enumerate(display_indices):
        if grid_num not in target_grid_set:
            opponent_seed_crop_indices.add(crop_idx)

    print(f"  Labeled grid: {len(target_crop_indices)} target / "
          f"{len(opponent_seed_crop_indices)} opponent-seed (not-picked)")

    target_refs_seed = X_scaled[list(target_crop_indices)]
    if opponent_seed_crop_indices:
        opponent_refs_seed = X_scaled[list(opponent_seed_crop_indices)]
    else:
        dists = np.full(n_total, np.inf)
        for tf in target_refs_seed:
            dists = np.minimum(dists, np.linalg.norm(X_scaled - tf, axis=1))
        n_force = max(3, n_total // 5)
        far_idx = np.argsort(dists)[::-1][:n_force]
        opponent_seed_crop_indices = set(int(i) for i in far_idx
                                          if int(i) not in target_crop_indices)
        opponent_refs_seed = X_scaled[list(opponent_seed_crop_indices)]
        print(f"  No not-picked grid samples - synthesized "
              f"{len(opponent_seed_crop_indices)} opponent seeds from farthest crops")

    # ── Classify with margin ──
    labels = np.full(n_total, -1, dtype=int)  # -1 = ambiguous
    confidences = np.zeros(n_total, dtype=float)

    for i in range(n_total):
        d_t = float(np.min(np.linalg.norm(target_refs_seed - X_scaled[i], axis=1)))
        d_o = float(np.min(np.linalg.norm(opponent_refs_seed - X_scaled[i], axis=1)))
        total = d_t + d_o + 1e-9
        if d_t <= d_o:
            conf = 1.0 - d_t / total
            tentative = 0
        else:
            conf = 1.0 - d_o / total
            tentative = 1
        confidences[i] = conf
        if conf >= classify_margin:
            labels[i] = tentative
        # else: stays -1 (ambiguous)

    # Force user-labeled crops to their explicit label, conf=1.
    for idx in target_crop_indices:
        labels[idx] = 0
        confidences[idx] = 1.0
    for idx in opponent_seed_crop_indices:
        labels[idx] = 1
        confidences[idx] = 1.0

    # ── Build final refs from confident classifications ──
    target_confident = [i for i in range(n_total)
                        if labels[i] == 0 and confidences[i] >= ref_confidence_min]
    opponent_confident = [i for i in range(n_total)
                          if labels[i] == 1 and confidences[i] >= ref_confidence_min]

    target_refs = X_scaled[target_confident] if target_confident else target_refs_seed
    opponent_refs = X_scaled[opponent_confident] if opponent_confident else opponent_refs_seed

    target_centroid = target_refs.mean(axis=0)
    opponent_centroid = opponent_refs.mean(axis=0)

    n_target = int((labels == 0).sum())
    n_opponent = int((labels == 1).sum())
    n_ambiguous = int((labels == -1).sum())

    _show_verification(crops, labels, confidences, target_crop_indices)

    print(f"\n[OK] Calibration: {len(target_crop_indices)} labeled target / "
          f"{len(opponent_seed_crop_indices)} labeled opponent")
    print(f"   Margin:       classify >= {classify_margin:.2f}  "
          f"refs >= {ref_confidence_min:.2f}")
    print(f"   Final refs:   target={len(target_refs)}  opponent={len(opponent_refs)}")
    print(f"   Classified:   target={n_target} ({n_target/n_total*100:.0f}%)  "
          f"opponent={n_opponent} ({n_opponent/n_total*100:.0f}%)  "
          f"ambiguous={n_ambiguous} ({n_ambiguous/n_total*100:.0f}%)")
    if n_opponent == 0:
        print("   WARN: zero opponents classified - check that opponent jerseys"
              " were present among NOT-picked grid samples.")
    if n_ambiguous > n_total * 0.5:
        print("   NOTE: more than half ambiguous - consider lowering classify_margin,"
              " or labeling more diverse target samples.")

    return {
        "target_refs": target_refs,
        "opponent_refs": opponent_refs,
        "target_centroid": target_centroid,
        "opponent_centroid": opponent_centroid,
        "scaler": scaler,
        "target_cluster": 0,
        "n_clusters": 2,
        "cluster_sizes": {0: n_target, 1: n_opponent, -1: n_ambiguous},
        "n_crops_total": n_total,
        "n_labeled_target": len(target_crop_indices),
        "n_labeled_opponent": len(opponent_seed_crop_indices),
        "classify_margin": classify_margin,
    }


def _show_verification(crops, labels, confidences, labeled_indices,
                       n_per_row=10):
    """3-row verification grid: target / ambiguous / opponent.

    Within each row, low-confidence crops appear FIRST so the user can
    immediately spot borderline cases that may be mislabeled.
    """
    rows_spec = [
        ("YOUR TEAM (target)", 0, "#2ECC71"),
        ("AMBIGUOUS (review)", -1, "#888888"),
        ("OPPONENT", 1, "#E74C3C"),
    ]

    fig, axes = plt.subplots(len(rows_spec), n_per_row,
                             figsize=(2.4 * n_per_row, 2.6 * len(rows_spec)))
    if len(rows_spec) == 1:
        axes = axes[np.newaxis, :]

    for row, (team_name, team_label, color) in enumerate(rows_spec):
        team_indices = np.where(labels == team_label)[0]
        if len(team_indices) == 0:
            show = np.array([], dtype=int)
        else:
            # Sort by confidence ASC → least-confident shown first.
            order = np.argsort(confidences[team_indices])
            ordered = team_indices[order]
            # If the team is huge, sample evenly across the confidence spectrum
            # but always include the worst few.
            if len(ordered) <= n_per_row:
                show = ordered
            else:
                worst_n = min(n_per_row // 2, len(ordered))
                rest_pool = ordered[worst_n:]
                rest_sample = rest_pool[np.linspace(0, len(rest_pool) - 1,
                                                    n_per_row - worst_n,
                                                    dtype=int)]
                show = np.concatenate([ordered[:worst_n], rest_sample])

        for j in range(n_per_row):
            ax = axes[row, j]
            if j < len(show):
                idx = int(show[j])
                crop = crops[idx]
                ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                conf = confidences[idx]
                # Show confidence on the crop title (small).
                ax.set_title(f"c={conf:.2f}", fontsize=8, pad=2)

                if idx in labeled_indices:
                    edge = "gold"; ew = 4
                else:
                    edge = color; ew = 2
                for spine in ax.spines.values():
                    spine.set_edgecolor(edge); spine.set_linewidth(ew)
            else:
                ax.set_facecolor("#f0f0f0")
                for spine in ax.spines.values():
                    spine.set_edgecolor("#dddddd"); spine.set_linewidth(1)
            ax.set_xticks([]); ax.set_yticks([])

            if j == 0:
                ax.set_ylabel(team_name, fontsize=11, fontweight="bold",
                              color=color, rotation=0, labelpad=70, ha="right",
                              va="center")

    n_t = int((labels == 0).sum())
    n_a = int((labels == -1).sum())
    n_o = int((labels == 1).sum())
    plt.suptitle(
        f"VERIFICATION  -  target={n_t}  ambiguous={n_a}  opponent={n_o}\n"
        "Gold borders = your labels. Low-confidence crops shown first per row.",
        fontsize=12, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.show()


def classify_person(torso_crop, calibration, overlay_crop_mask=None):
    """
    Classify a person as target team, opponent, or ambiguous.

    Uses k-NN: computes minimum distance to any target reference sample
    vs minimum distance to any opponent reference sample.
    This handles intra-class variety much better than single centroid.

    Falls back to centroid distance if reference sets are not available
    (backward compatibility).

    Args:
        overlay_crop_mask: overlay mask for the torso region (1=clean, 0=overlay).
            Passed through to extract_torso_features so overlay text
            pixels are excluded from the color histogram.

    Returns: (role, confidence)
      - role: "target", "opponent", or "ambiguous"
      - confidence: 0.0 to 1.0
    """
    if calibration is None:
        return "ambiguous", 0.0

    features = extract_torso_features(torso_crop, overlay_crop_mask)
    if features is None:
        return "ambiguous", 0.0

    X = calibration["scaler"].transform([features])[0]

    # k-NN: min distance to reference sets
    target_refs = calibration.get("target_refs")
    opponent_refs = calibration.get("opponent_refs")
    if target_refs is not None and opponent_refs is not None and len(opponent_refs) > 0:
        d_target = float(np.min(np.linalg.norm(target_refs - X, axis=1)))
        d_opponent = float(np.min(np.linalg.norm(opponent_refs - X, axis=1)))
    else:
        # Fallback: centroid distance
        d_target = np.linalg.norm(X - calibration["target_centroid"])
        d_opponent = np.linalg.norm(X - calibration["opponent_centroid"])

    # Confidence: ratio of distance difference
    total = d_target + d_opponent + 1e-9
    if d_target < d_opponent:
        confidence = 1.0 - (d_target / total)
        role = "target"
    else:
        confidence = 1.0 - (d_opponent / total)
        role = "opponent"

    # Low confidence -> ambiguous. Threshold uses the calibration's
    # classify_margin (set in build_calibration); falls back to 0.58.
    threshold = calibration.get("classify_margin", 0.58)
    if confidence < threshold:
        return "ambiguous", round(confidence, 3)

    return role, round(confidence, 3)


# ═══════════════════════════════════════════════════════════════════════════
# Legacy compatibility
# ═══════════════════════════════════════════════════════════════════════════

def discover_clusters(*args, **kwargs):
    """Deprecated. Use collect_samples() + show_samples() instead."""
    print("⚠️  discover_clusters() is deprecated.")
    print("   Use the new 3-step flow:")
    print("   1. sample_data = collect_samples(...)")
    print("   2. show_samples(sample_data)")
    print("   3. calibration = build_calibration(sample_data, MY_TEAM)")
    return None


def finalize_calibration(*args, **kwargs):
    """Deprecated. Use build_calibration() instead."""
    print("⚠️  finalize_calibration() is deprecated. Use build_calibration().")
    return None


### Module: pipeline

Pass 1 (fast zoom-level scan) and Pass 2 (team-aware extraction).
Reuses helpers / calibration defined above via shared notebook scope.


In [ ]:
# pipeline — Pass 1 fast scan + Pass 2 team-aware extraction
import cv2
import numpy as np
from tqdm import tqdm


# ── Default parameters ────────────────────────────────────────────────
# Each param: WHAT it controls → effect of ↑ (raise) / ↓ (lower)
DEFAULT_PARAMS = {
    # YOLO confidence for person detection.
    # ↑ fewer detections, miss distant players  ↓ more, more FPs
    "PERSON_CONFIDENCE": 0.45,

    # Minimum persons-in-frame to keep it.
    # ↑ group shots only  ↓ accepts solo shots
    "MIN_PERSONS": 1,

    # Minimum (largest bbox area / frame area). Camera zoom gate.
    # ↑ only extreme close-ups  ↓ accepts wide shots
    "MIN_MAX_PERSON_AREA_RATIO": 0.015,

    # Sharpness flag threshold (0-1). Below = flagged "motion blur",
    # still kept as a candidate.
    "MIN_SHARPNESS": 0.15,

    # Hard blur cutoff — frames below this are dropped entirely.
    # Main junk filter for motion blur.
    "MOTION_BLUR_SHARPNESS_MIN": 0.06,

    # Soft pitch-green filter (target-team frames bypass it).
    "ENABLE_PITCH_GREEN_FILTER": True,
    "PITCH_ROI_Y_START": 0.40,
    "MIN_PITCH_GREEN_RATIO": 0.04,

    # Pass 1 scan stride (frames). ↑ faster scan, may miss brief zooms.
    "SCAN_INTERVAL": 5,

    # Min consecutive quality frames to form a "segment".
    "MIN_SEGMENT_FRAMES": 2,

    # Consecutive bad frames allowed inside a segment before it ends.
    "SEGMENT_GAP_TOLERANCE": 3,
}


def pass1_fast_scan(video_path, yolo_model, device, params=None, max_frames=None):
    """Pass 1: scan every Nth frame, build a zoom timeline, group into segments."""
    p = {**DEFAULT_PARAMS, **(params or {})}

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_area = w * h

    scan_limit = max_frames if max_frames else total
    interval = p["SCAN_INTERVAL"]

    video_info = {"fps": fps, "total_frames": total, "width": w,
                  "height": h, "frame_area": frame_area}

    zoom_timeline = []
    batch_frames, batch_nums = [], []
    batch_size = 32

    pbar = tqdm(total=scan_limit, desc="Pass 1: Scanning zoom levels", unit="fr")
    frame_num = 0

    while True:
        ret, frame = cap.read()
        if not ret or frame_num >= scan_limit:
            break
        pbar.update(1)

        if frame_num % interval != 0:
            frame_num += 1
            continue

        batch_frames.append(frame)
        batch_nums.append(frame_num)

        if len(batch_frames) >= batch_size:
            _process_scan_batch(yolo_model, batch_frames, batch_nums,
                                zoom_timeline, frame_area, device, p)
            batch_frames, batch_nums = [], []

        frame_num += 1

    if batch_frames:
        _process_scan_batch(yolo_model, batch_frames, batch_nums,
                            zoom_timeline, frame_area, device, p)
    pbar.close()
    cap.release()

    segments = _find_segments(zoom_timeline, p)
    return segments, zoom_timeline, video_info


def _process_scan_batch(model, frames, frame_nums, timeline,
                        frame_area, device, params):
    """Score one batch of scan frames into (frame_num, max_area_ratio, n_persons)."""
    results = model.predict(frames, classes=[0], conf=params["PERSON_CONFIDENCE"],
                            device=device, verbose=False)
    for fn, res in zip(frame_nums, results):
        n_persons = len(res.boxes) if res.boxes is not None else 0
        max_ratio = 0.0
        if n_persons > 0:
            areas = []
            for box in res.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                areas.append((x2 - x1) * (y2 - y1) / frame_area)
            max_ratio = max(areas)
        timeline.append((fn, max_ratio, n_persons))


def _find_segments(zoom_timeline, params):
    """Find contiguous quality segments, allowing up to SEGMENT_GAP_TOLERANCE
    consecutive bad frames inside one segment before closing it."""
    min_ratio = params["MIN_MAX_PERSON_AREA_RATIO"]
    min_persons = params["MIN_PERSONS"]
    min_seg = params["MIN_SEGMENT_FRAMES"]
    gap_tol = params["SEGMENT_GAP_TOLERANCE"]

    segments = []
    current = []
    bad_streak = 0

    for fn, ratio, n_p in zoom_timeline:
        is_good = ratio >= min_ratio and n_p >= min_persons
        if is_good:
            current.append((fn, ratio, n_p))
            bad_streak = 0
        else:
            bad_streak += 1
            if bad_streak > gap_tol:
                if len(current) >= min_seg:
                    segments.append(current)
                current = []
                bad_streak = 0

    if len(current) >= min_seg:
        segments.append(current)

    return segments


def pass2_extract(video_path, segments, yolo_model, device,
                  calibration, overlay_mask, video_info, params=None,
                  max_frames=None, replay_region_norm=None):
    """Pass 2: walk each quality segment, classify each frame, score it."""
    p = {**DEFAULT_PARAMS, **(params or {})}
    fps = video_info["fps"]
    frame_area = video_info["frame_area"]
    frame_h = video_info["height"]
    frame_w = video_info["width"]
    total = video_info["total_frames"]
    scan_interval = p["SCAN_INTERVAL"]

    cap = cv2.VideoCapture(str(video_path))
    candidates = []
    stats = {
        "segments_processed": 0,
        "frames_analyzed": 0,
        "skipped_pitch": 0,
        "skipped_blurry": 0,
        "skipped_no_person": 0,
        "team_target": 0,
        "team_opponent": 0,
        "team_mixed": 0,
        "team_ambiguous": 0,
    }

    for seg_idx, segment in enumerate(tqdm(segments, desc="Pass 2: Team-aware extraction")):
        seg_start = segment[0][0]
        seg_end = segment[-1][0] + scan_interval
        seg_duration = (seg_end - seg_start) / fps

        cap.set(cv2.CAP_PROP_POS_FRAMES, seg_start)
        seg_candidates = []

        for fn in range(seg_start, min(seg_end, total)):
            ret, frame = cap.read()
            if not ret:
                break
            stats["frames_analyzed"] += 1

            # Tag (do not reject) replay / TV-graphics frames so we can
            # dedup them using the rugby-inset region downstream.
            is_replay = is_graphics_frame(frame)

            pitch_green = compute_pitch_green_ratio(frame, p["PITCH_ROI_Y_START"])

            detections = detect_persons(yolo_model, frame, device,
                                        p["PERSON_CONFIDENCE"])

            if len(detections) < p["MIN_PERSONS"]:
                # No persons + no pitch → almost certainly graphics/crowd.
                if p["ENABLE_PITCH_GREEN_FILTER"] and pitch_green < p["MIN_PITCH_GREEN_RATIO"]:
                    stats["skipped_pitch"] += 1
                    continue
                stats["skipped_no_person"] += 1
                continue

            fg_detections = filter_foreground_players(detections, frame_h, frame_w)

            max_person_ratio = max(d["area"] / frame_area for d in fg_detections)
            if max_person_ratio < p["MIN_MAX_PERSON_AREA_RATIO"]:
                continue

            n_target, n_opponent, n_ambiguous = 0, 0, 0
            for det in fg_detections:
                torso, status, torso_ov_mask = extract_torso_crop(
                    frame, det["bbox"], overlay_mask)
                if status == "ok" and torso is not None:
                    team, conf = classify_person(torso, calibration, torso_ov_mask)
                    det["team"] = team
                    det["team_conf"] = conf
                else:
                    det["team"] = "ambiguous"
                    det["team_conf"] = 0.0

                if det["team"] == "target":
                    n_target += 1
                elif det["team"] == "opponent":
                    n_opponent += 1
                else:
                    n_ambiguous += 1

            n_total = n_target + n_opponent
            category, shot_type = _categorize_frame(
                n_target, n_opponent, n_ambiguous, max_person_ratio
            )

            # Soft pitch filter: keep target-bearing frames regardless.
            if p["ENABLE_PITCH_GREEN_FILTER"]:
                has_target = n_target > 0
                has_large_person = max_person_ratio > 0.06
                if (not has_target and not has_large_person
                        and pitch_green < p["MIN_PITCH_GREEN_RATIO"]):
                    stats["skipped_pitch"] += 1
                    continue

            target_dets = [d for d in fg_detections if d.get("team") == "target"]
            sharpness_dets = target_dets if target_dets else fg_detections
            sharpness = compute_player_sharpness(frame, sharpness_dets, overlay_mask)

            if sharpness < p["MOTION_BLUR_SHARPNESS_MIN"]:
                stats["skipped_blurry"] += 1
                continue

            is_motion_blur = sharpness < p["MIN_SHARPNESS"]

            score = _compute_team_score(
                sharpness, fg_detections, frame_area, category,
                n_target, n_opponent, max_person_ratio,
                pitch_green
            )

            max_target_ratio = 0.0
            target_coverage = 0.0
            if target_dets:
                max_target_ratio = max(d["area"] / frame_area for d in target_dets)
                target_coverage = sum(d["area"] for d in target_dets) / frame_area

            team_dominance = n_target / max(n_total, 1)

            # pHash for dedup. Replay frames → hash only the rugby inset so
            # changes in the ref-cam don't make them look different.
            if is_replay and replay_region_norm is not None:
                phash = compute_phash(frame, region_norm=replay_region_norm)
            else:
                phash = compute_phash(frame)

            timestamp_sec = fn / fps
            seg_candidates.append({
                "frame_num": fn,
                "timestamp_sec": round(timestamp_sec, 3),
                "timestamp_hms": fmt_timestamp(timestamp_sec),
                "n_persons": len(fg_detections),
                "n_target": n_target,
                "n_opponent": n_opponent,
                "n_ambiguous": n_ambiguous,
                "team_dominance": round(team_dominance, 3),
                "category": category,
                "shot_type": shot_type,
                "sharpness": sharpness,
                "is_motion_blur": is_motion_blur,
                "max_person_area_ratio": round(max_person_ratio, 4),
                "max_target_area_ratio": round(max_target_ratio, 4),
                "target_coverage": round(target_coverage, 4),
                "pitch_green_ratio": round(pitch_green, 4),
                "score": round(score, 4),
                "segment_idx": seg_idx,
                "segment_duration": round(seg_duration, 1),
                "is_replay": bool(is_replay),
                "phash": phash,
            })

        stats["segments_processed"] += 1
        candidates.extend(seg_candidates)

    cap.release()

    for c in candidates:
        cat = c["category"]
        if "target" in cat:
            stats["team_target"] += 1
        elif "opponent" in cat:
            stats["team_opponent"] += 1
        elif cat == "mixed":
            stats["team_mixed"] += 1
        else:
            stats["team_ambiguous"] += 1

    candidates.sort(key=lambda x: x["timestamp_sec"])
    return candidates, stats


def _categorize_frame(n_target, n_opponent, n_ambiguous, max_person_ratio):
    """Bucket a frame by team composition × shot type."""
    n_total = n_target + n_opponent
    shot_type = get_shot_type(max_person_ratio)

    if n_total == 0:
        return f"ambiguous_{shot_type}", shot_type

    target_ratio = n_target / n_total

    if target_ratio >= 0.5:
        return f"target_{shot_type}", shot_type
    elif target_ratio > 0:
        return "mixed", shot_type
    else:
        return f"opponent_{shot_type}", shot_type


def _compute_team_score(sharpness, detections, frame_area, category,
                        n_target, n_opponent, max_person_ratio,
                        pitch_green=0.0):
    """Team-aware quality score. pitch_green = soft bonus, not a gate."""
    target_dets = [d for d in detections if d.get("team") == "target"]
    max_target_ratio = (max(d["area"] / frame_area for d in target_dets)
                        if target_dets else 0.0)

    pitch_bonus = min(pitch_green / 0.10, 1.0) * 0.05

    if "target" in category:
        score = (
            sharpness * 0.25 +
            max_target_ratio * 0.25 +
            min(n_target / 4, 1.0) * 0.20 +
            max_person_ratio * 0.15 +
            0.10 +
            pitch_bonus
        )
    elif category == "mixed":
        score = (
            sharpness * 0.30 +
            max_person_ratio * 0.25 +
            min((n_target + n_opponent) / 5, 1.0) * 0.15 +
            (n_target / max(n_target + n_opponent, 1)) * 0.15 +
            0.05 +
            pitch_bonus
        )
    else:
        score = (
            sharpness * 0.40 +
            max_person_ratio * 0.30 +
            min(len(detections) / 5, 1.0) * 0.15 +
            pitch_bonus
        )

    return round(score, 4)


def dedup_candidates(candidates, max_hash_dist=6, time_window_sec=3):
    """Drop near-duplicate frames using pHash distance within a time window.

    Walks candidates in timestamp order; for each frame, compare its pHash
    only to already-kept frames within the last `time_window_sec`. If the
    minimum Hamming distance is below `max_hash_dist`, the frame is dropped
    (the earlier one wins keep priority).

    For replay-flagged candidates the pHash was computed on the match-inset
    region, so this dedup correctly catches replays that only differ in the
    secondary (ref) inset.

    Args:
        candidates: list of candidate dicts produced by pass2_extract.
            Must contain 'timestamp_sec' and 'phash' fields.
        max_hash_dist: Hamming distance below which two frames count as
            duplicates (typical pHash range 0-64; 4-8 = perceptually same).
            Lower = stricter dedup, fewer drops.
        time_window_sec: only compare against frames within this lookback.
            Keep small (2-5s) so we only catch literal near-consecutive
            duplicates. Long windows (>10s) make the first kept frame an
            "anchor" that swallows every similar-looking frame in a long
            segment.

    Returns:
        (kept_list, n_dropped)
    """
    if not candidates:
        return candidates, 0
    ordered = sorted(candidates, key=lambda c: c["timestamp_sec"])
    kept = []
    dropped = 0
    for c in ordered:
        ph = c.get("phash")
        if ph is None:
            kept.append(c)
            continue
        is_dup = False
        for prev in reversed(kept):
            if c["timestamp_sec"] - prev["timestamp_sec"] > time_window_sec:
                break
            prev_ph = prev.get("phash")
            if prev_ph is None:
                continue
            if (ph - prev_ph) <= max_hash_dist:
                is_dup = True
                break
        if is_dup:
            dropped += 1
        else:
            kept.append(c)
    return kept, dropped



### Module: selection

Quota-based frame picker — balances the final dataset across team/shot buckets.


In [ ]:
# selection — quota-based frame picker
import numpy as np


def auto_target_frames(candidates, video_duration_sec,
                       frames_per_minute=5, max_candidate_ratio=0.6,
                       floor=50, ceiling=600):
    """Heuristic TARGET_FRAMES = clip(duration_min × frames_per_minute,
    floor, min(ceiling, len(candidates) × max_candidate_ratio))."""
    duration_min = video_duration_sec / 60
    base = int(duration_min * frames_per_minute)
    capped = min(base, int(len(candidates) * max_candidate_ratio), ceiling)
    result = max(capped, min(floor, len(candidates)))

    print(f"  Auto TARGET_FRAMES: {result}")
    print(f"    Video: {duration_min:.1f} min × {frames_per_minute}/min = {base} base")
    print(f"    Candidates: {len(candidates)} × {max_candidate_ratio} = {int(len(candidates)*max_candidate_ratio)} cap")
    print(f"    Bounds: [{floor}, {ceiling}]")

    return result


# Quotas (must sum to 1.0).
DEFAULT_QUOTA = {
    "target_closeup":   0.30,
    "target_medium":    0.25,
    "target_wide":      0.05,
    "mixed":            0.20,
    "opponent_closeup": 0.08,
    "opponent_medium":  0.07,
    "opponent_wide":    0.05,
}


def select_by_quota(candidates, total_target, quota=None, min_time_gap=1.5):
    """Pick `total_target` frames using per-category quotas and a time-gap diversity rule."""
    if not candidates:
        return [], {}

    q = quota or DEFAULT_QUOTA

    by_category = {}
    for c in candidates:
        quota_key = _map_to_quota_key(c["category"])
        by_category.setdefault(quota_key, []).append(c)

    for key in by_category:
        by_category[key].sort(key=lambda x: x["score"], reverse=True)

    selected = []
    selection_stats = {}

    for cat_key, fraction in sorted(q.items(), key=lambda x: -x[1]):
        n_want = max(1, int(total_target * fraction))
        pool = by_category.get(cat_key, [])

        picked = _pick_with_diversity(pool, n_want, min_time_gap)
        selected.extend(picked)
        selection_stats[cat_key] = len(picked)

    # Top-up from remaining if quotas under-filled.
    if len(selected) < total_target:
        selected_nums = {c["frame_num"] for c in selected}
        remaining = [c for c in candidates if c["frame_num"] not in selected_nums]
        remaining.sort(key=lambda x: x["score"], reverse=True)

        for c in remaining:
            if len(selected) >= total_target:
                break
            if all(abs(c["timestamp_sec"] - s["timestamp_sec"]) >= min_time_gap / 2
                   for s in selected):
                selected.append(c)

    selected.sort(key=lambda x: x["timestamp_sec"])
    return selected, selection_stats


def _map_to_quota_key(category):
    """Detailed category → quota bucket (ambiguous_* collapses to 'mixed')."""
    if category.startswith("ambiguous"):
        return "mixed"
    return category


def _pick_with_diversity(pool, n_want, min_gap_sec):
    """Greedy top-N from pool while enforcing a minimum time gap."""
    if not pool:
        return []

    picked = []
    for candidate in pool:
        if len(picked) >= n_want:
            break

        ts = candidate["timestamp_sec"]
        if all(abs(ts - p["timestamp_sec"]) >= min_gap_sec for p in picked):
            picked.append(candidate)

    return picked


def print_selection_summary(selected, selection_stats, total_candidates):
    """Pretty-print quota breakdown + score/sharpness ranges."""
    print(f"\n{'=' * 60}")
    print(f"  SELECTION RESULTS")
    print(f"{'=' * 60}")
    print(f"  Total candidates:  {total_candidates}")
    print(f"  Selected frames:   {len(selected)}")
    print(f"")
    print(f"  Per-category breakdown:")
    for cat, count in sorted(selection_stats.items()):
        pct = count / max(len(selected), 1) * 100
        bar = "#" * int(pct / 2)
        print(f"    {cat:25s}  {count:4d}  ({pct:5.1f}%)  {bar}")

    if selected:
        scores = [c["score"] for c in selected]
        sharp = [c["sharpness"] for c in selected]
        print(f"\n  Score range:       {min(scores):.3f} - {max(scores):.3f}")
        print(f"  Sharpness range:   {min(sharp):.3f} - {max(sharp):.3f}")
        print(f"  Time span:         {selected[0]['timestamp_hms']} - "
              f"{selected[-1]['timestamp_hms']}")
    print(f"{'=' * 60}")


## 3. Load Video & Model

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

PERSON_DETECTION_MODEL = "yolo11l.pt"

# JPEG quality for saved frames.
#   100 = lossless-ish (~2x file size, no measurable training benefit)
#   95  = visually lossless, preserves fine logo edges (~20-30 px logos)  ← default
#   90  = subtle ringing on sharp text/logo edges
#   85  = visible artifacts; only use if storage is tight
JPEG_QUALITY = 95

# Test mode only scans the first 2000 frames; production scans the whole video.
MAX_SCAN_FRAMES = 2000 if TEST_MODE else TOTAL_FRAMES

tf_input = input("Target frames (press Enter for AUTO, or type a number): ").strip()
TARGET_FRAMES = int(tf_input) if tf_input.isdigit() else None

mode_label = "TEST" if TEST_MODE else "PRODUCTION"
target_label = TARGET_FRAMES if TARGET_FRAMES is not None else "AUTO"
print(f"\nDuration: {TOTAL_FRAMES/FPS/60:.1f}min | {W}x{H} | {FPS:.0f}fps | {TOTAL_FRAMES:,} frames")
print(f"Mode: {mode_label} | Scan: {MAX_SCAN_FRAMES:,} | Target: {target_label}")

print(f"\nLoading {PERSON_DETECTION_MODEL}...")
yolo_model = YOLO(PERSON_DETECTION_MODEL)
print("Model loaded.")


## 4. Define Overlay Regions

Mark scoreboard / watermark / channel-logo areas as rectangles. Auto-detection is still available (`detect_static_overlays`) but manual boxes give cleaner masks for known broadcast layouts.


In [ ]:
# Grab a representative game frame for the picker.
cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, TOTAL_FRAMES // 3)
ret, sample_frame = cap.read()
cap.release()
assert ret, "Failed to read sample frame from video"

# Draw your overlay rectangles in the canvas below (scoreboard, watermarks,
# channel logos). Click "Done" when finished.
# To skip the picker and paste coords manually, comment the next line and set
#   OVERLAY_REGIONS = [(0.78, 0.00, 1.00, 0.18), (0.00, 0.84, 0.55, 1.00)]
OVERLAY_REGIONS = interactive_box_picker(
    sample_frame,
    prompt="Drag rectangles over overlays (scoreboard, BullsTV logo, etc.). Click Done."
)

print(f"Captured {len(OVERLAY_REGIONS)} overlay region(s):")
for r in OVERLAY_REGIONS:
    print(f"  ({r[0]:.3f}, {r[1]:.3f}, {r[2]:.3f}, {r[3]:.3f})")

overlay_mask, overlay_ratio = build_manual_overlay_mask(
    sample_frame.shape, OVERLAY_REGIONS, normalized=True
)
print(f"\nOverlay coverage: {overlay_ratio * 100:.1f}% masked")

visualize_overlay_with_grid(sample_frame, overlay_mask, grid_step=0.05)


## 4B. (Optional) Define Replay Match-Inset Region

Replays show the rugby footage inside a small inset; the rest is broadcast
graphics + a secondary cam (usually the referee). Two consecutive replay
frames often only differ in the ref-cam, fooling full-frame dedup.

If you want **smarter dedup on replays**, scrub through your video to find a
"Video Ref Replay" moment, note the timestamp in seconds, then enter it
below to seek there and draw the rugby-inset rectangle.

To **skip this step**, just press Enter at the prompt — dedup will fall back
to full-frame pHash everywhere (still works fine for most videos).


In [ ]:
ts_input = input(
    "Replay timestamp in seconds (e.g. 41.0) — or press Enter to skip: "
).strip()

if not ts_input:
    MATCH_REPLAY_REGION = None
    print("Skipped. MATCH_REPLAY_REGION = None — dedup will use full frame.")
else:
    try:
        ts_sec = float(ts_input)
        fn = int(ts_sec * FPS)
        if fn < 0 or fn >= TOTAL_FRAMES:
            raise ValueError(f"Timestamp out of range (video is {TOTAL_FRAMES/FPS:.1f}s)")

        cap = cv2.VideoCapture(str(VIDEO_PATH))
        cap.set(cv2.CAP_PROP_POS_FRAMES, fn)
        ret, replay_frame = cap.read()
        cap.release()
        if not ret:
            raise RuntimeError("Failed to read frame at that timestamp")

        print(f"Seeking to frame {fn} ({ts_sec:.3f}s). Draw the rugby-inset rectangle:")
        MATCH_REPLAY_REGION = interactive_box_picker(
            replay_frame,
            prompt="Drag ONE rectangle around the RUGBY MATCH inset. Click Done.",
            single_box=True,
        )
    except Exception as e:
        print(f"Could not seek/pick: {e}")
        print("Setting MATCH_REPLAY_REGION = None.")
        MATCH_REPLAY_REGION = None

if MATCH_REPLAY_REGION:
    x1, y1, x2, y2 = MATCH_REPLAY_REGION
    print(f"\nMATCH_REPLAY_REGION = ({x1:.3f}, {y1:.3f}, {x2:.3f}, {y2:.3f})")
else:
    print("\nMATCH_REPLAY_REGION = None")


## 5A. Collect & Display Torso Samples

This samples diverse player torso crops from the video.

**Look at the numbered grid** and write down which numbers show **YOUR team's jersey**.

In [ ]:
sample_data = collect_samples(
    VIDEO_PATH, yolo_model, DEVICE,
    overlay_mask=overlay_mask,
    n_sample_frames=80,
    n_display=24,
)

show_samples(sample_data)

## 5B. Label YOUR Team

Look at the numbered grid above. **Edit `MY_TEAM` below** with the numbers
that show YOUR team's jersey (Bradford Bulls = dark/black striped jersey).

You need **at least 3** numbers. More is better (5-8 ideal).

Example: `MY_TEAM = [0, 3, 5, 8, 11, 14]`

In [ ]:
# ✏️ Enter the grid numbers that show YOUR team (comma-separated)
raw = input("Enter grid numbers for YOUR team (e.g. 0,3,5,8): ")
MY_TEAM = [int(x.strip()) for x in raw.split(",") if x.strip().isdigit()]

if len(MY_TEAM) < 3:
    raise ValueError(f"Need at least 3 samples, got {len(MY_TEAM)}: {MY_TEAM}")

print(f"\nYour team labels: {MY_TEAM}")
calibration = build_calibration(sample_data, MY_TEAM)

# The verification grid below shows the result.
# Top row = your team, Bottom row = opponent.
# Gold borders = samples you labeled.
# If it looks wrong, re-run this cell with corrected numbers.

## 6. Pass 1 — Fast Scan

In [ ]:
segments, zoom_timeline, video_info = pass1_fast_scan(
    VIDEO_PATH, yolo_model, DEVICE, max_frames=MAX_SCAN_FRAMES
)

print(f"\nScanned: {len(zoom_timeline):,} frames")
print(f"Quality segments: {len(segments)}")
if segments:
    avg_len = np.mean([(s[-1][0]-s[0][0])/FPS for s in segments])
    print(f"Avg segment length: {avg_len:.1f}s")

## 7. Pass 2 — Team-Aware Extraction

In [ ]:
candidates, pass2_stats = pass2_extract(
    VIDEO_PATH, segments, yolo_model, DEVICE,
    calibration, overlay_mask, video_info,
    max_frames=MAX_SCAN_FRAMES,
    replay_region_norm=MATCH_REPLAY_REGION,
)

print(f"\nCandidates: {len(candidates)}")
print(f"  Target: {pass2_stats['team_target']} | Mixed: {pass2_stats['team_mixed']} | Opponent: {pass2_stats['team_opponent']}")
n_replay = sum(1 for c in candidates if c.get("is_replay"))
print(f"  Skipped: pitch={pass2_stats['skipped_pitch']}, blur={pass2_stats['skipped_blurry']}")
print(f"  Replay-flagged: {n_replay} ({n_replay/max(len(candidates),1)*100:.0f}%)")

## 7B. Dedup Near-Duplicate Frames

Adjacent frames inside replays / camera pauses can be near-identical. We
drop them here using pHash within a time window. Replay candidates use the
match-inset crop (configured in 4B), normal candidates use the full frame.


In [ ]:
# Drop near-duplicate frames using pHash.
# Tuning notes:
#   time_window_sec: lookback for dedup comparison.
#     2-3s  = catches literal consecutive-frame duplicates only (safe)
#     5-10s = also catches slow-motion / replay segments  (may over-prune)
#     >15s  = aggressive — risks pruning whole play sequences
#   max_hash_dist: 0-64 Hamming distance.
#     4-6  = perceptually-identical only
#     8-12 = also drops visually-similar (action progressing slowly)
n_before = len(candidates)
candidates, n_dropped = dedup_candidates(
    candidates,
    max_hash_dist=6,
    time_window_sec=3,
)
drop_pct = n_dropped / max(n_before, 1) * 100
print(f"Dedup dropped {n_dropped}/{n_before} ({drop_pct:.1f}%); "
      f"{len(candidates)} remain.")

if drop_pct > 70:
    print("WARN: dedup dropped >70% of candidates. Consider:")
    print("  - Increase max_hash_dist (8-10) so only true duplicates drop")
    print("  - Decrease time_window_sec further (1-2s) if many legit frames")
    print("    are being lost inside long action segments.")


## 8. Quota Selection

In **production mode**, `TARGET_FRAMES` is auto-calculated:
- Base: `video_duration_min × 5 frames/min`
- Capped: never more than 60% of candidates (quality filter)
- Bounds: min 50, max 600 per video

In [ ]:
# Auto-compute if not set (production mode)
if TARGET_FRAMES is None:
    TARGET_FRAMES = auto_target_frames(
        candidates,
        video_duration_sec=TOTAL_FRAMES / FPS,
    )

selected, sel_stats = select_by_quota(candidates, TARGET_FRAMES)
print_selection_summary(selected, sel_stats, len(candidates))

## 9. Save Frames & Metadata

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
rows = []

for idx, meta in enumerate(tqdm(selected, desc="Saving")):
    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret: continue

    fname = f"{MATCH_ID}_{meta['frame_num']:06d}_{meta['timestamp_sec']:.3f}s.jpg"
    cv2.imwrite(str(FRAMES_OUTPUT_DIR / fname), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

    row = {k: v for k, v in meta.items() if k != "phash"}
    row.update({"filename": fname, "match_id": MATCH_ID,
                "member": MEMBER_NAME, "extracted_at": datetime.now().isoformat()})
    rows.append(row)

cap.release()

df = pd.DataFrame(rows)
csv_path = METADATA_DIR / f"{MATCH_ID}_v3_index.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved {len(df)} frames to: {FRAMES_OUTPUT_DIR}\nMetadata: {csv_path}")

## 10. Preview by Category

In [ ]:
for cat in sorted(df["category"].unique()):
    cat_df = df[df["category"] == cat].sort_values("score", ascending=False)
    n_show = min(4, len(cat_df))
    if n_show == 0: continue

    fig, axes = plt.subplots(1, n_show, figsize=(5*n_show, 5))
    if n_show == 1: axes = [axes]

    for i, (_, row) in enumerate(cat_df.head(n_show).iterrows()):
        fpath = FRAMES_OUTPUT_DIR / row["filename"]
        if fpath.exists():
            img = cv2.imread(str(fpath))
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[i].set_title(f"{row['timestamp_hms']} | {row['score']:.2f}\ntarget={row['n_target']} opp={row['n_opponent']}", fontsize=8)
        axes[i].axis("off")

    plt.suptitle(f"{cat.upper()} ({len(cat_df)} frames)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

print(f"\nCategory distribution:\n{df['category'].value_counts().to_string()}")
print(f"\nShot type distribution:\n{df['shot_type'].value_counts().to_string()}")

## 11. Quality Charts

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0,0].hist(df["sharpness"], bins=25, color="steelblue", edgecolor="white")
axes[0,0].set_title("Sharpness")
axes[0,1].hist(df["score"], bins=25, color="forestgreen", edgecolor="white")
axes[0,1].set_title("Quality Score")
axes[0,2].hist(df["team_dominance"], bins=20, color="coral", edgecolor="white")
axes[0,2].set_title("Team Dominance")
axes[1,0].hist(df["max_person_area_ratio"], bins=25, color="mediumpurple", edgecolor="white")
axes[1,0].set_title("Largest Person Size")
axes[1,1].hist(df["timestamp_sec"]/60, bins=25, color="goldenrod", edgecolor="white")
axes[1,1].set_title("Temporal Distribution (min)")
cat_counts = df["category"].value_counts()
axes[1,2].barh(cat_counts.index, cat_counts.values, color="teal")
axes[1,2].set_title("Frames per Category")
plt.suptitle(f"{MATCH_ID} — {len(df)} frames", fontsize=14)
plt.tight_layout()
plt.show()

## 12. Upload to Roboflow (Optional)

In [ ]:
import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass.getpass("Roboflow API Key: ")
ROBOFLOW_PROJECT = "kit-sponsor-logos"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace().project(ROBOFLOW_PROJECT)

for fp in tqdm(sorted(FRAMES_OUTPUT_DIR.glob(f"{MATCH_ID}_*.jpg")), desc="Uploading"):
    try:
        project.upload(
            image_path=str(fp),
            split="train",
            tag_names=[MATCH_ID, MEMBER_NAME, "v3"],
        )
    except Exception as e:
        print(f"Error: {e}")
        break

print("Upload complete.")
